In [ ]:
import streamlit as st
from pathlib import Path

# Page configuration
st.set_page_config(
    page_title="File Manager",
    page_icon="📁",
    layout="centered"
)

# Custom CSS for better styling
st.markdown("""
<style>
    .main-header {
        text-align: center;
        padding: 1rem 0 2rem 0;
    }
    .operation-card {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 1.5rem;
        border-radius: 10px;
        color: white;
        margin-bottom: 1rem;
    }
    .success-box {
        padding: 1rem;
        border-radius: 8px;
        background-color: #d4edda;
        border: 1px solid #c3e6cb;
        color: #155724;
    }
    .error-box {
        padding: 1rem;
        border-radius: 8px;
        background-color: #f8d7da;
        border: 1px solid #f5c6cb;
        color: #721c24;
    }
    .file-content {
        background-color: #f8f9fa;
        padding: 1rem;
        border-radius: 8px;
        border: 1px solid #dee2e6;
        font-family: monospace;
        white-space: pre-wrap;
        word-wrap: break-word;
    }
    .stButton > button {
        width: 100%;
        border-radius: 8px;
        padding: 0.5rem 1rem;
        font-weight: 600;
    }
</style>
""", unsafe_allow_html=True)

# Header
st.markdown("<h1 class='main-header'>📁 File Manager</h1>", unsafe_allow_html=True)
st.markdown("<p style='text-align: center; color: #6c757d; margin-bottom: 2rem;'>Create, Read, Update & Delete files with ease</p>", unsafe_allow_html=True)

# Sidebar for operation selection
st.sidebar.markdown("## 🛠️ Operations")
operation = st.sidebar.radio(
    "Select an operation:",
    ["📝 Create File", "📖 Read File", "✏️ Update File", "🗑️ Delete File"],
    label_visibility="collapsed"
)

st.sidebar.markdown("---")
st.sidebar.markdown("### 📊 Quick Stats")

# Count files in current directory
current_files = list(Path(".").glob("*"))
file_count = len([f for f in current_files if f.is_file()])
st.sidebar.metric("Files in Directory", file_count)

st.sidebar.markdown("---")
st.sidebar.markdown(
    "<p style='color: #6c757d; font-size: 0.8rem;'>Built with ❤️ using Streamlit</p>",
    unsafe_allow_html=True
)

# Helper functions
def show_success(message):
    st.markdown(f"<div class='success-box'>✅ {message}</div>", unsafe_allow_html=True)

def show_error(message):
    st.markdown(f"<div class='error-box'>❌ {message}</div>", unsafe_allow_html=True)

def get_existing_files():
    """Get list of files in current directory for dropdown"""
    return [f.name for f in Path(".").glob("*") if f.is_file()]

# Main content area
st.markdown("---")

# CREATE FILE
if operation == "📝 Create File":
    st.markdown("### 📝 Create a New File")
    
    col1, col2 = st.columns([2, 1])
    with col1:
        filename = st.text_input("File Name", placeholder="example.txt")
    with col2:
        st.markdown("<br>", unsafe_allow_html=True)
        
    content = st.text_area(
        "File Content",
        placeholder="Enter the content you want to write...",
        height=200
    )
    
    if st.button("🚀 Create File", type="primary"):
        if not filename:
            show_error("Please enter a file name")
        else:
            try:
                path = Path(filename)
                if path.exists():
                    show_error(f"File '{filename}' already exists!")
                else:
                    with open(path, "w") as f:
                        f.write(content)
                    show_success(f"File '{filename}' created successfully!")
                    st.balloons()
            except Exception as e:
                show_error(f"An error occurred: {e}")

# READ FILE
elif operation == "📖 Read File":
    st.markdown("### 📖 Read a File")
    
    existing_files = get_existing_files()
    
    if existing_files:
        filename = st.selectbox("Select a file to read:", existing_files)
        
        if st.button("📂 Read File", type="primary"):
            try:
                path = Path(filename)
                with open(path, "r") as f:
                    content = f.read()
                
                st.markdown("#### 📄 File Content:")
                st.markdown(f"<div class='file-content'>{content if content else '<em>File is empty</em>'}</div>", unsafe_allow_html=True)
                
                # File info
                col1, col2, col3 = st.columns(3)
                with col1:
                    st.metric("Size", f"{path.stat().st_size} bytes")
                with col2:
                    st.metric("Lines", len(content.splitlines()))
                with col3:
                    st.metric("Characters", len(content))
                    
            except Exception as e:
                show_error(f"An error occurred: {e}")
    else:
        st.info("📭 No files found in the current directory. Create one first!")

# UPDATE FILE
elif operation == "✏️ Update File":
    st.markdown("### ✏️ Update a File")
    
    existing_files = get_existing_files()
    
    if existing_files:
        filename = st.selectbox("Select a file to update:", existing_files)
        
        update_type = st.radio(
            "Update Type:",
            ["🔄 Rename", "➕ Append", "📝 Overwrite"],
            horizontal=True
        )
        
        if update_type == "🔄 Rename":
            new_name = st.text_input("New File Name", placeholder="new_name.txt")
            
            if st.button("🔄 Rename File", type="primary"):
                if not new_name:
                    show_error("Please enter a new file name")
                else:
                    try:
                        path = Path(filename)
                        new_path = Path(new_name)
                        if new_path.exists():
                            show_error(f"File '{new_name}' already exists!")
                        else:
                            path.rename(new_path)
                            show_success(f"File renamed from '{filename}' to '{new_name}'!")
                            st.rerun()
                    except Exception as e:
                        show_error(f"An error occurred: {e}")
        
        elif update_type == "➕ Append":
            # Show current content
            with st.expander("📄 View Current Content"):
                try:
                    with open(Path(filename), "r") as f:
                        st.code(f.read() or "File is empty")
                except:
                    st.warning("Could not read file")
            
            append_content = st.text_area(
                "Content to Append",
                placeholder="Enter content to add...",
                height=150
            )
            
            if st.button("➕ Append to File", type="primary"):
                try:
                    with open(Path(filename), "a") as f:
                        f.write("\n" + append_content)
                    show_success(f"Content appended to '{filename}'!")
                except Exception as e:
                    show_error(f"An error occurred: {e}")
        
        elif update_type == "📝 Overwrite":
            st.warning("⚠️ This will replace all existing content!")
            
            new_content = st.text_area(
                "New Content",
                placeholder="Enter new content...",
                height=200
            )
            
            if st.button("📝 Overwrite File", type="primary"):
                try:
                    with open(Path(filename), "w") as f:
                        f.write(new_content)
                    show_success(f"File '{filename}' overwritten successfully!")
                except Exception as e:
                    show_error(f"An error occurred: {e}")
    else:
        st.info("📭 No files found in the current directory. Create one first!")

# DELETE FILE
elif operation == "🗑️ Delete File":
    st.markdown("### 🗑️ Delete a File")
    
    existing_files = get_existing_files()
    
    if existing_files:
        filename = st.selectbox("Select a file to delete:", existing_files)
        
        st.warning(f"⚠️ Are you sure you want to delete '{filename}'? This action cannot be undone!")
        
        col1, col2 = st.columns(2)
        with col1:
            if st.button("🗑️ Delete File", type="primary"):
                try:
                    Path(filename).unlink()
                    show_success(f"File '{filename}' deleted successfully!")
                    st.rerun()
                except Exception as e:
                    show_error(f"An error occurred: {e}")
        with col2:
            if st.button("❌ Cancel"):
                st.info("Deletion cancelled.")
    else:
        st.info("📭 No files found in the current directory.")

# Footer
st.markdown("---")
st.markdown(
    "<p style='text-align: center; color: #6c757d;'>📁 File Manager v1.0 | Made with Streamlit</p>",
    unsafe_allow_html=True
)

2026-06-20 03:37:06.387 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 03:37:06.390 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 03:37:07.213 
  command:

    streamlit run C:\Users\dutta\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-06-20 03:37:07.215 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 03:37:07.217 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 03:37:07.242 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 03:37:07.246 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

DeltaGenerator()

In [ ]:
!jupyter nbconvert --to script app.ipynb

[NbConvertApp] Converting notebook app.ipynb to script
[NbConvertApp] Writing 8970 bytes to app.py


In [ ]:
!jupyter nbconvert --to script Untitled10.ipynb

[NbConvertApp] Converting notebook Untitled10.ipynb to script
[NbConvertApp] Writing 8792 bytes to Untitled10.py


In [ ]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!streamlit run Untitled10.py